In [ ]:
import pandas as pd
import calendar


# Load dataset
file_path = r"C:\Users\USER\Documents\Python\Project\cleaned_cafc_data.csv"
df = pd.read_csv(file_path)

# Convert Date column to datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Removing "-" from the column titles
df.columns = (
    df.columns
    .str.strip()
    .str.replace("_", " ")
    .str.title()
)
# Renaming the  Received Type column as Submission Channel for better understanding
df = df.rename(columns={"Received Type": "Submission Channel"})


# Check data types
print(df.info())

# Check missing values sum
print(df.isnull().sum())

print((df.isnull().sum() / len(df)) * 100)

# Replace "_-_" first (for age range)
df = df.replace("_-_", "-", regex=True)

# Replace all underscores with space
df = df.replace("_", " ", regex=True)

# drop all countries that is not Canada
df = df[df["Country"] == "Canada"]

# Preview data
print(df.head())

df.to_csv(r"C:\Users\USER\Documents\Python\Project\cleaned_final_data.csv", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 350146 entries, 0 to 350145
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Id                  350146 non-null  int64         
 1   Date                350146 non-null  datetime64[us]
 2   Submission Channel  350146 non-null  str           
 3   Country             350146 non-null  str           
 4   Province State      350146 non-null  str           
 5   Category            350146 non-null  str           
 6   Method              331904 non-null  str           
 7   Gender              259167 non-null  str           
 8   Language            260137 non-null  str           
 9   Age Range           245470 non-null  str           
 10  Complaint Type      350146 non-null  str           
 11  Victims             350146 non-null  int64         
 12  Loss                350146 non-null  float64       
 13  Age Median          245470 non-null  flo

In [52]:
# Filter victims
victims_df = df[df["Complaint Type"] == "Victim"]

# Total amount of money lost
total_loss = df["Loss"].sum()
print(f"Total Loss: ${total_loss:,.2f}")

Total Loss: $2,078,198,422.83


In [49]:
# Total Loss by province from highest to lowest
loss_by_province = df.groupby("Province State")["Loss"].sum().sort_values(ascending=False)
print("\nTop 5 Provinces by Loss:")
print(loss_by_province.head())


Top 5 Provinces by Loss:
Province State
Ontario             1.006929e+09
British Columbia    3.536261e+08
Alberta             2.726937e+08
Quebec              2.388162e+08
Manitoba            8.134554e+07
Name: Loss, dtype: float64


In [45]:
# 2. Common Fraud Threat from highest to lowet
counts = df["Category"].value_counts()

top_category = counts.idxmax()
top_count = counts.max()

print("\nMost Common Fraud Categories:")
print(f"{top_category} → {top_count}")


loss_by_category = df.groupby("Category")["Loss"].sum().sort_values(ascending=False)    # Fraud type with the most loss of money (Highest to lowest)
print("\nFraud Types with Highest Loss:")
print(loss_by_category.head())



Most Common Fraud Categories:
Identity Fraud → 71376

Fraud Types with Highest Loss:
Category
Investments       1.069056e+09
Spear Phishing    2.361093e+08
Romance           2.264038e+08
Job               9.142597e+07
Service           6.908817e+07
Name: Loss, dtype: float64


In [53]:
# 3. Most Suspectible victims by Age Group

# Analyze age
age_counts = victims_df["Age Range"].value_counts()
print("\nMost Affected Age Groups:")
print(age_counts.head())



Most Affected Age Groups:
Age Range
30-39    33697
20-29    28556
40-49    28442
60-69    24509
50-59    24159
Name: count, dtype: int64


In [51]:
# 4. Most Suspectible victim by Gender
gender_counts = victims_df["Gender"].value_counts().reset_index()
gender_counts.columns = ["Gender", "Count"]

print("\nMost Affected Gender Groups:")
print(gender_counts.head())


Most Affected Gender Groups:
              Gender  Count
0               Male  83672
1             Female  82327
2            Unknown    250
3  Prefer Not To Say    243
4              Other    136


In [ ]:
# 5. Fraud Trend over time

trend = (
    df.groupby(df["Date"].dt.to_period("M"))
    .size()
    .reset_index(name="Complaint Count")
)
print("\nFraud Trends Over Time:")
print(trend.head())


Fraud Trends Over Time:
      Date  Complaint Count
0  2021-01             6586
1  2021-02             7474
2  2021-03             8195
3  2021-04             6342
4  2021-05             6001


In [54]:
# 6. Most Damaging Fraud per person
damage_df = victims_df[victims_df["Victims"] > 0].copy()
damage_df["Loss per Victim"] = damage_df["Loss"] / damage_df["Victims"]

damage = (
    damage_df.groupby("Category")["Loss per Victim"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
print("\nMost Damaging Fraud per Victim:")
print(damage.head())


Most Damaging Fraud per Victim:
              Category  Loss per Victim
0       Spear Phishing     84144.457662
1          Investments     77490.302826
2              Romance     58111.864073
3  Foreign Money Offer     56924.537738
4            Timeshare     53923.684697


In [55]:
# 7. Months with Fraud Spikes

monthly_spike = (
    victims_df.groupby("Month")
    .size()
    .reset_index(name="Complaint Count")
    .sort_values("Month")
)

monthly_spike["Month Name"] = monthly_spike["Month"].apply(lambda x: calendar.month_name[x])

# Sorting by Complaint Count from highest to lowest
monthly_spike = monthly_spike.sort_values(by="Complaint Count", ascending=False).reset_index(drop=True)
print("\nMonths with Highest Fraud Cases:")
print(monthly_spike.head())


Months with Highest Fraud Cases:
   Month  Complaint Count Month Name
0      3            19078      March
1      1            16921    January
2      5            16612        May
3      6            16028       June
4      2            16006   February
